In [6]:
import re

def parse_expression(expr):
    predicate, args = expr.split("(", 1)
    args = args.rstrip(")")
    arguments = [arg.strip() for arg in args.split(",")]
    return (predicate.strip(), *arguments)


def is_variable(x):
    return x.isalpha() and x[0].islower()


def occurs(var, expr):
    if var == expr:
        return True
    if isinstance(expr, tuple):
        return any(occurs(var, arg) for arg in expr[1:])
    return False


def apply_subst(expr, subst):
    if isinstance(expr, str):
        while expr in subst:
            expr = subst[expr]
        return expr

    if isinstance(expr, tuple):
        return (expr[0],) + tuple(
            apply_subst(arg, subst) for arg in expr[1:]
        )

    return expr


def unify(x, y):
    if x == y:
        return {}

    if isinstance(x, str) and is_variable(x):
        if occurs(x, y):
            return None
        return {x: y}

    if isinstance(y, str) and is_variable(y):
        if occurs(y, x):
            return None
        return {y: x}

    if not isinstance(x, tuple) or not isinstance(y, tuple):
        return None

    if x[0] != y[0] or len(x) != len(y):
        return None

    subst = {}

    for a, b in zip(x[1:], y[1:]):
        a = apply_subst(a, subst)
        b = apply_subst(b, subst)

        s = unify(a, b)

        if s is None:
            return None

        for key in subst:
            subst[key] = apply_subst(subst[key], s)

        subst.update(s)

    return subst


# Main program
expr1 = input("Enter expression1: ")
expr2 = input("Enter expression2: ")

e1 = parse_expression(expr1)
e2 = parse_expression(expr2)

result = unify(e1, e2)

if result is None:
    print("Unification: FAILURE")
elif not result:
    print("Unification: NIL")
else:
    print("Unification: SUCCESS")
    print("Substitution:")
    for var, value in result.items():
        print(f"{var} -> {value}")

Enter expression1: Eats (y, Apple)
Enter expression2: Eats (Rahul, z)
Unification: SUCCESS
Substitution:
y -> Rahul
z -> Apple
